# Lezione 6: NumPy and Pandas

### Contatti
[matteo.caldana@polimi.it](mailto:matteo.caldana@polimi.it).

### Un veloce throwback al terminale

Anche in Colab abbiamo il terminale, se volessimo potremmo fare tutto dal esso (ma è molto scomodo vedere il contenuto dei files senza un editor di testo).

## NumPy

NumPy è una libreria di algebra lineare che contiene tutte le funzionalità di Matlab (esclusi i widget aggiuntivi). Visto che nei corsi di numerica vi spiegano già Matlab nei dettagli, questa è solo una carrellata delle peculiarità di Numpy. Se siete interessati a sapere come convertire comandi Matlab in NumPy trovate qui una comoda guida:

https://numpy.org/doc/stable/user/numpy-for-matlab-users.html

Oppure leggete la guida completa

https://numpy.org/doc/stable/user/basics.html

In [ ]:
import numpy as np

### Creare Array

Creiamo un elemento di $R$, un array 0D

In [ ]:
a0d = np.array(0.32)
a0d

Creiamo un elmento di $R^{m_1}$, un array 1D

In [ ]:
a1d = np.array([0.1, 0.2, 0.3])
a1d

Creiamo un elemento di $R^{{m_1}\times{m_2}}$, un array 2D

In [ ]:
a2d = np.array([[0.1, 0.2, 0.3], [-0.4, -0.5, 0.6]])
a2d

Possiamo creare un qualsiare elemento di $R^{{m_1}\times{m_2}\times ... \times{m_n}}$, un array $n$-D.

Ci sono le stesse funzioni per crearli di Matlab
- `zeros`
- `ones`
- `eye`
- `linspace`
- `diag`

Ed alcune nuove come `empty` per avere della memoria non-inizializzata (più veloce, l'OS non deve fare pulizia)

In [ ]:
a3d = np.empty((2, 3, 4))
a3d

Come controllare le dimensioni di un array?

In [ ]:
print("Quali sono m_1, ..., m_n?: ", a3d.shape)
print("Che dimensione ha?: ", a3d.ndim)
print("Quanti elementi in totale?", a3d.size, "(prodotto di m_1 * ... * m_n)")

Numpy è veloce poichè implementato in C, per questo motivo gli elementi degli array, come in C, hanno un tipo e un numero di bit che viene utilizzato per questo tipo

In [ ]:
a2d.dtype 

Il tipo può essere imposto durante la creazione (con un kwarg) o cambiato a posteriori (cast)

In [ ]:
b = np.array([1, 2, 3], dtype=np.float32)
print(b)
b = np.array([1, 2, 3], dtype=np.int64)
print(b)

a1d.astype(np.int32)

### Operazioni per asse

La maggior parte delle operazioni matematiche, es. `np.sum`, possono essere applicate su tutto l'array o su un sotto insieme di assi.

In [ ]:
a2d

Somma totale

In [ ]:
np.sum(a2d)

Somma colonna per colonna (la sommatoria è effettuata per ogni valore nel `range(shape[0])`)
$$\sum_{i}^{m_1} a_{ij}$$

In [ ]:
np.sum(a2d, axis=0)

Somma riga per riga (la sommatoria è effettuata per ogni valore nel `range(shape[1])`)
$$\sum_{j}^{m_2} a_{ij}$$

In [ ]:
np.sum(a2d, axis=1)

Possiamo passare ad `axis` uno o più indici (tramite una tupla). 

In [ ]:
np.sum(a2d, axis=(0, 1))

### Copy and View

Per poter cambiare un elemento di un array, tipo

In [ ]:
b[1] = 99
b

L'oggetto `b[1]` non può essere una copia di `b`, deve poter accedere alla memoria di `b` per cambiarlo!

Esistono due tipi di array:
- `copy`: possiedono la memoria a cui stanno facendo riferimento (es. `b`)
- `view`: **non** possiedono la memoria a cui stanno facendo riferimento (es. `b[1]`)

Come controllare?

In [ ]:
c = b[1:2]
print("`b`          è una copia?", b.base is None)
print("`c = b[1:2]` è una copia?", c.base is None)

### Reshape

Uno dei motivi che rende NumPy così veloce è che dato un array, tutti i suoi elementi sono **contigui in memoria**, questo significa che NumPy garantisce che tutti gli elementi dell'array, dal primo all'ultimo, sono ordinati e fisicamente vicini sul PC.

Per **fisicamente vicini** intendo letteralmente che sono salvati in transistor che sono vicini tra di loro nella RAM in termini di nanometri. Questa proprietà è ereditata direttamente dal fatto che gli array di C
```c
void buffer[] = ...
```
hanno questa caratteristica. Indipendetenmente dalla dimensione `ndim`, NumPy salva sempre i dati in un singolo array di C.

Questo significa che **i seguenti** due array hanno la stessa rapprentazione in memoria anche se hanno dimensione diversa

In [ ]:
a1d = np.array([1, 2, 3, 4, 5, 6])
a2d = np.array([[1, 2, 3], [4, 5, 6]])

print(a1d)
print(a2d)

Capita quindi spesso che sia utile passare da una rappresentazione all'altra senza fare una `copy` ma usando invece una `view`.

Ecco l'importanza del comando `reshape`

In [ ]:
a2d_view = a1d.reshape((2, 3))
print(a2d_view)
print(a2d_view.base is None)

Naturalmente, `reshape` funziona solo se e solo se il prodotto dei termini della tupla passata è pari alla `size` dell'array da cui stiamo chiamando il metodo `reshape`.

#### Memory layout: "row major" detto anche "right layout" 

<img src="./assets/row-major.png" alt="row-major" style="width: 300px;"/>

Il modo in cui `reshape` "spiattella" (da 2D a 1D) o "avvolge" (da 1D a 2D) la memoria è abbastanza intuitivo
- la "spiattella" leggendo la matrice prima da sinitra verson destra e poi dall'alto al basso (come si legge una pagina di un libro).
- la "avvolge" esattamente al contrario, spezzando in una nuova righa ogni volta che raggiunge il numero di caratteri pari al numero di colonne della matrice

Questa convenzione di chiama "row major" perchè le righe `a2d[i, :]` rimangono intatte per ogni `i`. 

Questo funziona per 1D e 2D, ma cosa succede per array a più alta dimensione?

Il ragionamento è ricorsivo: per esempio per un array 3D `a3d[:, :, :]`, lo si divide in matrici `a3d[j, :, :]` e si "spiattella" ogni matrice e poi si uniscono insieme le matrici spiattellate.

Questa convenzione di chiama "right layout" perchè l'asse più a destra `a3d[j, i, :]` rimane intatto per ogni `i` e `j`. 

### Indexing

L'indexing funziona esattamente come per le liste, e similmente a Matlab

In [ ]:
# elemento sulla seconda riga e terza colonna
print(a2d[1, 2])
# tutte gli elmenti della prima riga in posizione pari: primo (0), terzo (2)
print(a2d[0, ::2])

Si possono passare liste/array come indici, ricordatevi che anche i **numeri negativi valgono**.

Se l'array ha dimensione $n>1$ tutte le liste/array che vengono passate devono avere la stessa lunghezza o seguire delle regole molto particolare, vedete https://numpy.org/doc/stable/user/basics.indexing.html.

In [ ]:
idxs = np.array([-1, -3])
a1d[idxs]

Caso 2D, gli indici vengono trattati come tuple di indexing 

In [ ]:
idxs_rows = np.array([1, 1])
idxs_cols = np.array([1, 2])
# estraiamo l=2 elementi (dove l==idxs_rows.size==idxs_cols.size)
# alle posizioni (1, 1) e (1, 2)
a2d[idxs_rows, idxs_cols]

Un altro modo per estrarre elementi è con una **maschera booleana**, cioè un array della stessa esatta *shape* dell'array che indica su quali elementi stiamo lavorando

In [ ]:
a = np.arange(56).reshape((7, 8))
a

In [ ]:
boolean_mask = a > 31
print("boolean mask:\n", boolean_mask)
a[a>31] += 100
a

`np.where` è una funzione che potenzia le maschere booleane 

In [ ]:
# se passiamo un solo valore, la maschera booleana
# troviamo gli indici dove la maschera booleana è vera
print(np.where(a>31))

# se passiamo tre valori: 
# - la maschera booleana
# - valore caso VERO
# - valore caso FALSO
# applichiamo un IF-ELSE con la maschera booleana
print(np.where(a>31, -1, 7))

#### Problemi non solo da Matematici

A differenza di Matlab dove tutto è una matrice (e quindi un vettore è una matrice con una riga), in NumPy siamo più vicini a quello che succede in matematica, in particolare, un elemento di 
- $R^n$
- $R^{1 \times n}$
- $R^{n \times 1}$

Sono tre cose ben diverse

In [ ]:
a1d = np.array([1, 2, 3])
a2d_col = a1d.reshape((3, 1))
a2d_row = a1d.reshape((1, 3))
print("1D vector")
print(a1d)
print("2D column matrix")
print(a2d_col)
print("2D row matrix")
print(a2d_row)

### Broadcasting

NumPy è veloce solo fino a quando riusciamo ad evitare **for** loops.

I **for** loops vanno evitati come la peste.

Sarete tanto più bravi come programmatori NumPy, tanto più riuscirete a evitare di scrivere **for**.

Il **broadcasting** dice che possiamo applicare un operazione a due array che non hanno la stessa shape se 

aggiungendo dimensione 1 all'array con il minor `ndims` a sinistra fino ad averne come quello con dimensione maggiore, si hanno dimensioni corrispontenti uguali o una delle due è 1.

#### Esempio 1

<img src="./assets/broadcasting_1.png" alt="b1" style="width: 400px;"/>


In [ ]:
# a.shape =       (3,)
# b.shape = () -> (1,)

a = np.array([1.0, 2.0, 3.0])
b = 2.0
a + b

#### Esempio 2: aggiungere una traslazione a una lista di punti in 3D

<img src="./assets/broadcasting_2.png" alt="b2" style="width: 500px;"/>

In [ ]:
a = np.array([[ 0.0,  0.0,  0.0],
              [10.0, 10.0, 10.0],
              [20.0, 20.0, 20.0],
              [30.0, 30.0, 30.0]])
b = np.array([1.0, 2.0, 3.0])

# a.shape =         (4, 3)
# b.shape = (3,) -> (1, 3)

a + b

b = np.array([1.0, 2.0, 3.0, 4.0])

# a.shape =         (4, 3)
# b.shape = (4,) -> (1, 4)

# NON compatibili!

#### Esempio 3
<img src="./assets/broadcasting_3.png" alt="b3" style="width: 500px;"/>

In [ ]:
a = np.array([0.0, 10.0, 20.0, 30.0])
b = np.array([1.0, 2.0, 3.0])

# a[:, np.newaxis] == a.reshape((4, 1)) 
# è utile perchè funziona per qualunque dimensione di `a`
# mentre con il reshape se la dimensione != 4 abbiamo errore 

# a.shape =         (4, 1)
# b.shape = (3,) -> (1, 3)

a[:, np.newaxis] * b

## Pandas

In data, c'è un esempio dei files che potremmo aver ottenuto lo scorso laboratorio grazie a Gemini.

Iniziamo a trovare quali sono tutti i CSV nella cartella.

In [ ]:
import os

if not os.path.exists("data"):
  !git clone https://github.com/AIM-mate/corso-python-26.git
  !cp -R corso-python-26/lezione-6/data .

In [ ]:
files = [file for file in os.listdir("data") if file.endswith("csv")]
files

Carichiamo tutti i files con Pandas

In [ ]:
import pandas as pd

dfs = [pd.read_csv(f"./data/{file}") for file in files]
dfs[0]

Concateniamo tutti i DataFrame (i files), in un DataFrame unico

In [ ]:
df = pd.concat(dfs)

Controllare le categorie di valori in una colonna

In [ ]:
df["city"].unique()

L'**indice** di un DataFrame è una serie di etichette che identificano ciascuna riga. Le etichette possono essere numeri interi, stringhe o qualsiasi altro tipo **hashable**. L'indice viene utilizzato per l'accesso e l'allineamento basati sulle etichette e può essere consultato o modificato tramite questo attributo.

In [ ]:
df.index

In [ ]:
df.index.unique()

Si accede all'indice con `loc[]`, le parentesi `[]` sono riservate per le colonne

In [ ]:
df.loc[0]

A meno che abbiamo un indice particolarmente significativo, spesso si cerca di mantenere l'indice "pulito", resettandolo al `range(0, len(df))`

In [ ]:
df = df.reset_index(drop=True)
df.index

Controlliamo i tipi delle colonne

In [ ]:
df.info()

Alcune colonne sono `object` (stringhe) quando in realtà vorremmo fossero numeri (es. la temperatura che ha il simbolo di grado centigrado).

Per sistemare queste colonne dobbiamo applicare una funzione per trasformare la stringa in `float`.

In [ ]:
def parse(string):
    # Perche' vale la pena usare try/except qui?
    # controllare se una stringa puo' essere trasformata
    # in float e' estremamente costoso e complesso
    # Notate inoltre come la parte di codice nel try sia la piu'
    # piccola possibile e catturiamo solo ValueError
    first_part = string.split(" ")[0]
    try:
        return float(first_part)
    except ValueError:
        return float('nan')

for col in ["T Media", "T min", "T max", "Umidità UR%", "Vento Max", "Precip.", "Raffica"]:
  df[col] = df[col].apply(parse)
  
df

In [ ]:
df.describe()

Creiamo la data

In [ ]:
MONTH_TABLE = {
    "Agosto": 8,
    "Settembre": 9,
    "Ottobre": 10,
    "Novembre": 11,
    "Dicembre": 12
}

def create_date(row):
  return f"{row['year']:d}-{MONTH_TABLE[row['month']]:02}-{row['Giorno G']:02}"

df["data"] = df.apply(create_date, axis=1)
df["data"] = pd.to_datetime(df['data'])

df["data"]

### Temperatura media a Milano

In [ ]:
# I dati di Milano
df_milano = df[df["city"] == "Milano"]

# Calcoliamo la media
df_milano["T Media"].mean()

#### Precipitazioni a Milano il 25 Dicembre 2025

In [ ]:
row = df[(df["city"] == "Milano") & (df["data"] == "2025-12-25")]
row["Precip."]

#### Confronta la temperatura media di tutte le città

In [ ]:
df.groupby('city')['T Media'].mean()

#### Confronta la temperatura minima e massima di tutte le città

In [ ]:
# mappiamo una colonna a un'operazione
df.groupby('city').agg({"T min": "min", "T max": "max"})

#### Crea una tabella con le città sulle colonne e la temperatura media come valore per ogni data

<img src="./assets/reshaping_pivot.png" alt="melt" style="width: 500px;"/>

In [ ]:
df_pivot = df.pivot(index='data', columns='city', values='T Media')
df_pivot

#### Da una tabella pivot torniamo indietro

<img src="./assets/reshaping_melt.png" alt="melt" style="width: 500px;"/>

In [ ]:
df_pivot.columns

In [ ]:
df_pivot.reset_index().columns

In [ ]:
df_pivot.reset_index().melt(id_vars='data', value_name='temperature')

#### Calcoliamo la media delle temperature per ogni mese e città

In [ ]:
df.pivot_table(
    index='month',      # Cosa vogliamo sulle righe
    columns='city',     # Cosa vogliamo sulle colonne
    values='T Media',     # Il dato da analizzare
    aggfunc='mean',     # Come aggregare i dati (poteva essere 'sum', 'count', 'max')
).round(1) # Arrotondiamo per rendere la tabella leggibile

#### Temperatura media a 7 giorni 

In [ ]:
df["temp_7day_avg"] =  df.sort_values('data').groupby('city')['T Media'].transform(lambda x: x.rolling(7).mean())

In [ ]:
df.pivot(index='data', columns='city', values='temp_7day_avg').plot(title="7 Day Average Temp")